In [0]:

from pyspark.sql import functions as F

# Ler a tabela Bronze
clientes = spark.table("first_catalog.default.bronze_clientes")

# Colunas de negócio, excluindo metadados de ingestão
colunas_negocio = [
    "id_cliente",
    "nome_cliente",
    "segmento",
    "cidade",
    "estado",
    "pais",
    "mercado",
    "regiao"
]

# Limpar espaços nas extremidades dos campos de texto
silver_clientes = clientes.select(
    *[
        F.trim(F.col(c)).alias(c)
        for c in colunas_negocio
    ]
)

# Validar a chave de negócio
duplicados_clientes = (
    silver_clientes
    .groupBy("id_cliente")
    .count()
    .filter(F.col("count") > 1)
)

print("Total de clientes:", silver_clientes.count())
print("IDs de cliente duplicados:", duplicados_clientes.count())

# Guardar em Delta
(
    silver_clientes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("first_catalog.default.silver_clientes")
)

display(silver_clientes.limit(10))

Total de clientes: 1590
IDs de cliente duplicados: 0


id_cliente,nome_cliente,segmento,cidade,estado,pais,mercado,regiao
RH-19495,Rick Hansen,Consumidor,New York City,New York,United States,US,East
JR-16210,Justin Ritter,Corporativo,Wollongong,New South Wales,Australia,APAC,Oceania
CR-12730,Craig Reiter,Consumidor,Brisbane,Queensland,Australia,APAC,Oceania
KM-16375,Katherine Murray,Outros,Berlin,Berlin,Germany,EU,Central
RH-9495,Rick Hansen,Consumidor,Dakar,Dakar,Senegal,Africa,Africa
JM-15655,Jim Mitchum,Corporativo,Sydney,New South Wales,Australia,APAC,Oceania
TS-21340,Toby Swindell,Consumidor,Porirua,Wellington,New Zealand,APAC,Oceania
MB-18085,Mick Brown,Consumidor,Hamilton,Waikato,New Zealand,APAC,Oceania
JW-15220,Jane Waco,Corporativo,Sacramento,California,United States,US,West
JH-15985,Joseph Holt,Consumidor,Concord,North Carolina,United States,US,South


In [0]:

from pyspark.sql import functions as F

# Ler a tabela Bronze
pedidos = spark.table("first_catalog.default.bronze_pedidos")

# Selecionar e limpar as colunas de negócio
colunas_pedidos = [
    "id_pedido",
    "data_pedido",
    "data_envio",
    "modo_envio",
    "prioridade_pedido"
]

pedidos_limpos = pedidos.select(
    *[
        F.trim(F.col(c)).alias(c)
        for c in colunas_pedidos
    ]
)

# Converter as datas de dd-MM-yyyy para DATE
pedidos_tipados = (
    pedidos_limpos
    .withColumn(
        "data_pedido",
        F.to_date(F.col("data_pedido"), "dd-MM-yyyy")
    )
    .withColumn(
        "data_envio",
        F.to_date(F.col("data_envio"), "dd-MM-yyyy")
    )
)

# Remover apenas duplicados exatos nas colunas de negócio
pedidos_deduplicados = pedidos_tipados.dropDuplicates(
    colunas_pedidos
)

# Calcular o número de dias entre pedido e envio
silver_pedidos = pedidos_deduplicados.withColumn(
    "dias_para_envio",
    F.datediff(
        F.col("data_envio"),
        F.col("data_pedido")
    )
)

# Verificações de qualidade
duplicados_id = (
    silver_pedidos
    .groupBy("id_pedido")
    .count()
    .filter(F.col("count") > 1)
)

datas_invalidas = silver_pedidos.filter(
    F.col("data_pedido").isNull()
    | F.col("data_envio").isNull()
)

envios_antes_do_pedido = silver_pedidos.filter(
    F.col("data_envio") < F.col("data_pedido")
)

print("Pedidos na Bronze:", pedidos.count())
print("Pedidos após deduplicação:", pedidos_deduplicados.count())
print("Duplicados exatos removidos:",
      pedidos.count() - pedidos_deduplicados.count())
print("IDs de pedido duplicados:", duplicados_id.count())
print("Registos com datas inválidas:", datas_invalidas.count())
print("Envios anteriores à data do pedido:",
      envios_antes_do_pedido.count())

# Guardar a tabela Silver
(
    silver_pedidos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("first_catalog.default.silver_pedidos")
)

display(silver_pedidos.limit(10))

Pedidos na Bronze: 25036
Pedidos após deduplicação: 25035
Duplicados exatos removidos: 1
IDs de pedido duplicados: 0
Registos com datas inválidas: 0
Envios anteriores à data do pedido: 0


id_pedido,data_pedido,data_envio,modo_envio,prioridade_pedido,dias_para_envio
CA-2012-124891,2012-07-31,2012-07-31,Mesmo Dia,Critico,0
IN-2013-77878,2013-02-05,2013-02-07,Segunda Classe,Critico,2
IN-2013-71249,2013-10-17,2013-10-18,Primeira Classe,Medio,1
ES-2013-1579342,2013-01-28,2013-01-30,Primeira Classe,Medio,2
SG-2013-4320,2013-11-05,2013-11-06,Mesmo Dia,Critico,1
IN-2013-42360,2013-06-28,2013-07-01,Segunda Classe,Critico,3
IN-2011-81826,2011-11-07,2011-11-09,Primeira Classe,Critico,2
IN-2012-86369,2012-04-14,2012-04-18,Classe Padrao,Alto,4
CA-2014-135909,2014-10-14,2014-10-21,Classe Padrao,Baixo,7
CA-2012-116638,2012-01-28,2012-01-31,Segunda Classe,Critico,3


Transformar os produtos

In [0]:

from pyspark.sql import functions as F

# Ler os dados Bronze
produtos = spark.table("first_catalog.default.bronze_produtos")

# Definir as colunas de negócio
colunas_produtos = [
    "id_produto",
    "categoria",
    "subcategoria",
    "nome_produto"
]

# Limpar espaços no início e no fim dos campos
produtos_limpos = produtos.select(
    *[
        F.trim(F.col(c)).alias(c)
        for c in colunas_produtos
    ]
)

# Remover duplicados exatos
produtos_deduplicados = produtos_limpos.dropDuplicates(
    colunas_produtos
)

# Verificar identificadores repetidos
duplicados_id = (
    produtos_deduplicados
    .groupBy("id_produto")
    .count()
    .filter(F.col("count") > 1)
)

# Verificar campos nulos ou vazios
campos_invalidos = produtos_deduplicados.filter(
    F.col("id_produto").isNull()
    | (F.col("id_produto") == "")
    | F.col("categoria").isNull()
    | (F.col("categoria") == "")
    | F.col("subcategoria").isNull()
    | (F.col("subcategoria") == "")
    | F.col("nome_produto").isNull()
    | (F.col("nome_produto") == "")
)

# Guardar o resultado Silver
(
    produtos_deduplicados.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("first_catalog.default.silver_produtos")
)

print("Produtos na Bronze:", produtos.count())
print("Produtos após deduplicação:", produtos_deduplicados.count())
print(
    "Duplicados exatos removidos:",
    produtos.count() - produtos_deduplicados.count()
)
print("IDs de produto duplicados:", duplicados_id.count())
print("Registos com campos inválidos:", campos_invalidos.count())

display(produtos_deduplicados.limit(10))

Produtos na Bronze: 10293
Produtos após deduplicação: 10292
Duplicados exatos removidos: 1
IDs de produto duplicados: 0
Registos com campos inválidos: 0


id_produto,categoria,subcategoria,nome_produto
TEC-AC-10003033,Tecnologia,Accessories,Plantronics CS510 - Over-the-Head monaural Wireless Headset System
FUR-CH-10003950,Moveis,Chairs,"Novimex Executive Leather Armchair, Black"
TEC-PH-10004664,Tecnologia,Phones,"Nokia Smart Phone, with Caller ID"
TEC-PH-10004583,Tecnologia,Phones,"Motorola Smart Phone, Cordless"
TEC-SHA-10000501,Tecnologia,Copiers,"Sharp Wireless Fax, High-Speed"
TEC-PH-10000030,Tecnologia,Phones,"Samsung Smart Phone, with Caller ID"
FUR-CH-10004050,Moveis,Chairs,"Novimex Executive Leather Armchair, Adjustable"
FUR-TA-10002958,Moveis,Tables,"Chromcraft Conference Table, Fully Assembled"
OFF-BI-10003527,Material de Escritorio,Binders,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind
FUR-TA-10000198,Moveis,Tables,Chromcraft Bull-Nose Wood Oval Conference Tables & Bases


Transformar as vendas

In [0]:

from pyspark.sql import functions as F

# Ler a tabela Bronze
vendas = spark.table("first_catalog.default.bronze_vendas")

# Limpar identificadores e campos numéricos
vendas_limpa = (
    vendas
    .select(
        F.trim("pedido").alias("pedido"),
        F.trim("cliente").alias("cliente"),
        F.trim("produto").alias("produto"),
        F.trim("valor_venda").alias("valor_venda_raw"),
        F.trim("quantidade_vendida").alias("quantidade_raw"),
        F.trim("custo_envio").alias("custo_envio_raw")
    )
)

# Normalizar a vírgula decimal e converter os tipos
vendas_tipadas = (
    vendas_limpa
    .withColumn(
        "valor_venda",
        F.regexp_replace("valor_venda_raw", ",", ".")
         .cast("decimal(18,3)")
    )
    .withColumn(
        "quantidade_vendida",
        F.col("quantidade_raw").cast("int")
    )
    .withColumn(
        "custo_envio",
        F.regexp_replace("custo_envio_raw", ",", ".")
         .cast("decimal(18,3)")
    )
)

# Identificar falhas de conversão
# Só consideramos falha quando o texto original não é nulo/vazio
falhas_conversao = vendas_tipadas.filter(
    (
        F.col("valor_venda_raw").isNotNull()
        & (F.col("valor_venda_raw") != "")
        & F.col("valor_venda").isNull()
    )
    | (
        F.col("quantidade_raw").isNotNull()
        & (F.col("quantidade_raw") != "")
        & F.col("quantidade_vendida").isNull()
    )
    | (
        F.col("custo_envio_raw").isNotNull()
        & (F.col("custo_envio_raw") != "")
        & F.col("custo_envio").isNull()
    )
)

# Verificar campos obrigatórios
campos_invalidos = vendas_tipadas.filter(
    F.col("pedido").isNull() | (F.col("pedido") == "")
    | F.col("cliente").isNull() | (F.col("cliente") == "")
    | F.col("produto").isNull() | (F.col("produto") == "")
    | F.col("valor_venda").isNull()
    | F.col("quantidade_vendida").isNull()
    | F.col("custo_envio").isNull()
)

# Verificar valores não positivos
valores_nao_positivos = vendas_tipadas.filter(
    (F.col("valor_venda") <= 0)
    | (F.col("quantidade_vendida") <= 0)
    | (F.col("custo_envio") < 0)
)

# Criar a Silver com as colunas finais
silver_vendas = vendas_tipadas.select(
    "pedido",
    "cliente",
    "produto",
    "valor_venda",
    "quantidade_vendida",
    "custo_envio"
)

# Guardar em Delta
(
    silver_vendas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("first_catalog.default.silver_vendas")
)

# Apresentar métricas de qualidade
print("Vendas na Bronze:", vendas.count())
print("Vendas na Silver:", silver_vendas.count())
print("Falhas de conversão:", falhas_conversao.count())
print("Registos com campos obrigatórios inválidos:",
      campos_invalidos.count())
print("Registos com valores não positivos:",
      valores_nao_positivos.count())

display(silver_vendas.limit(10))

Vendas na Bronze: 51290
Vendas na Silver: 51290
Falhas de conversão: 0
Registos com campos obrigatórios inválidos: 0
Registos com valores não positivos: 0


pedido,cliente,produto,valor_venda,quantidade_vendida,custo_envio
CA-2012-124891,RH-19495,TEC-AC-10003033,2309.650,7,933.570
IN-2013-77878,JR-16210,FUR-CH-10003950,3709.395,9,923.630
IN-2013-71249,CR-12730,TEC-PH-10004664,5175.171,9,915.490
ES-2013-1579342,KM-16375,TEC-PH-10004583,2892.510,5,910.160
SG-2013-4320,RH-9495,TEC-SHA-10000501,2832.960,8,903.040
IN-2013-42360,JM-15655,TEC-PH-10000030,2862.675,5,897.350
IN-2011-81826,TS-21340,FUR-CH-10004050,1822.080,4,894.770
IN-2012-86369,MB-18085,FUR-TA-10002958,5244.840,6,878.380
CA-2014-135909,JW-15220,OFF-BI-10003527,5083.960,5,867.690
CA-2012-116638,JH-15985,FUR-TA-10000198,4297.644,13,865.740


In [0]:

from pyspark.sql import functions as F

clientes = spark.table("first_catalog.default.silver_clientes")
pedidos = spark.table("first_catalog.default.silver_pedidos")
produtos = spark.table("first_catalog.default.silver_produtos")
vendas = spark.table("first_catalog.default.silver_vendas")

# Conjuntos de identificadores válidos
ids_clientes = clientes.select(
    F.col("id_cliente").alias("cliente")
).distinct()

ids_pedidos = pedidos.select(
    F.col("id_pedido").alias("pedido")
).distinct()

ids_produtos = produtos.select(
    F.col("id_produto").alias("produto")
).distinct()

# Encontrar referências de vendas sem correspondência
clientes_em_falta = (
    vendas.select("cliente").distinct()
    .join(ids_clientes, on="cliente", how="left_anti")
)

pedidos_em_falta = (
    vendas.select("pedido").distinct()
    .join(ids_pedidos, on="pedido", how="left_anti")
)

produtos_em_falta = (
    vendas.select("produto").distinct()
    .join(ids_produtos, on="produto", how="left_anti")
)

print("Clientes em falta:", clientes_em_falta.count())
print("Pedidos em falta:", pedidos_em_falta.count())
print("Produtos em falta:", produtos_em_falta.count())

# Verificar que os IDs continuam únicos
print(
    "IDs de cliente duplicados:",
    clientes.groupBy("id_cliente").count()
    .filter(F.col("count") > 1).count()
)

print(
    "IDs de pedido duplicados:",
    pedidos.groupBy("id_pedido").count()
    .filter(F.col("count") > 1).count()
)

print(
    "IDs de produto duplicados:",
    produtos.groupBy("id_produto").count()
    .filter(F.col("count") > 1).count()
)

Clientes em falta: 0
Pedidos em falta: 0
Produtos em falta: 0
IDs de cliente duplicados: 0
IDs de pedido duplicados: 0
IDs de produto duplicados: 0
